# KrishiNexus · `krishi-whisper-bn-v1`: Bangla speech-to-text on Kaggle

Fine-tunes `openai/whisper-small` on Common Voice 13 Bengali, scores it once on a fixed test sample against the
zero-shot base model, and converts it to the int8 CTranslate2 format that the app's faster-whisper loads. The converted
copy is scored too, because that is the model the app runs. TRAINING_PLAN.md section 8c is the design.

**Data:** Mozilla Common Voice 13, Bengali (CC0), the normalized subset on Kaggle
`umongsain/common-voice-13-bengali-normalized`, with Common Voice's official train / dev / test splits.

**Pre-registered targets** (set before training):
- WER ≤ 0.45 and CER ≤ 0.15 on 2,000 seeded test clips, both below the zero-shot base model's;
- the int8 CTranslate2 copy's WER within 0.02 of the checkpoint on 300 of those clips.

**Before you run:** GPU T4, Internet on, and the dataset attached. Without a GPU the notebook runs a CPU smoke test on
a few clips; its scores are not results.

**Frozen-test rule:** the checkpoint is chosen on 500 dev clips. The test sample is read once, after that.

**Outputs (`/kaggle/working`):** `krishi-whisper-bn-v1/` (fp16 Hugging Face model and processor),
`krishi-whisper-bn-v1-ct2-int8/` (faster-whisper model), `eval_report.json`, `predictions_test.jsonl` and
`run_manifest.json`.


In [ ]:
%pip install -q -U "transformers>=5.22,<6" "accelerate>=1.0" faster-whisper librosa


In [ ]:
import glob, gc, json, os, platform, shutil, subprocess, sys, time, traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import librosa, numpy as np, torch, transformers

GPU = torch.cuda.is_available()
SMOKE = not GPU                     # CPU session: a few steps on a few clips, to exercise every cell; not a result
WORK = Path("/kaggle/working")
ROOT = Path(glob.glob("/kaggle/input/**/train.tsv", recursive=True)[0]).parent
BASE_MODEL = "openai/whisper-small"
OUT, CT2 = WORK / "krishi-whisper-bn-v1", WORK / "krishi-whisper-bn-v1-ct2-int8"
SEED = 20261109
SR, MAX_SECONDS = 16_000, 30
STEPS, BATCH, LR, WARMUP, EVAL_EVERY = (4, 2, 1e-5, 0, 2) if SMOKE else (3000, 16, 1e-5, 300, 500)
N_VAL, N_TEST, N_PARITY, N_TRAIN = (8, 16, 8, 32) if SMOKE else (500, 2000, 300, None)
TARGETS = {"wer": 0.45, "cer": 0.15, "ct2_wer_gap": 0.02}    # TRAINING_PLAN.md section 8c, set before training
DEVICE, DTYPE = ("cuda", torch.float16) if GPU else ("cpu", torch.float32)
torch.manual_seed(SEED)
print("data:", ROOT, "| GPU:", torch.cuda.get_device_name(0) if GPU else "none (CPU smoke test)", "| transformers", transformers.__version__)


In [ ]:
"""Helpers for the Bangla Whisper notebook: transcript normalization, WER/CER and seeded samples (no dependencies)."""
import random
import re
import unicodedata

PUNCT = re.compile(r"[।॥!\"#$%&'()*+,\-./:;<=>?@\[\\\]^_`{|}~‘’“”–—…]")


def normalize(text):
    """NFC, punctuation (including the Bengali danda) to spaces, whitespace collapsed."""
    return " ".join(PUNCT.sub(" ", unicodedata.normalize("NFC", text)).split())


def edit_distance(ref, hyp):
    """Levenshtein distance between two sequences."""
    prev = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        cur = [i] + [0] * len(hyp)
        for j, h in enumerate(hyp, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (r != h))
        prev = cur
    return prev[-1]


def error_rates(refs, hyps):
    """Corpus WER and CER: total edits over total reference words (characters, spaces excluded for CER)."""
    words = chars = word_edits = char_edits = 0
    for ref, hyp in zip(refs, hyps):
        r, h = normalize(ref), normalize(hyp)
        word_edits += edit_distance(r.split(), h.split())
        words += len(r.split())
        char_edits += edit_distance(r.replace(" ", ""), h.replace(" ", ""))
        chars += len(r.replace(" ", ""))
    return {"wer": round(word_edits / max(words, 1), 4), "cer": round(char_edits / max(chars, 1), 4), "words": words}


def sample(rows, n, seed):
    """A fixed random sample of n rows (all rows when n >= len(rows)), in a stable order."""
    rows = sorted(rows, key=lambda r: r["path"])
    return rows if n >= len(rows) else random.Random(seed).sample(rows, n)


In [ ]:
import csv
csv.field_size_limit(10**7)

def read_split(name):
    with open(ROOT / f"{name}.tsv", encoding="utf-8") as f:
        return [{"path": r["path"], "sentence": r["sentence"], "split": name} for r in csv.DictReader(f, delimiter="\t")]

TRAIN = read_split("train")
VAL = sample(read_split("dev"), N_VAL, SEED)
TEST = sample(read_split("test"), N_TEST, SEED + 1)
PARITY = TEST[:N_PARITY]
if N_TRAIN:
    TRAIN = sample(TRAIN, N_TRAIN, SEED)
print({"train": len(TRAIN), "val": len(VAL), "test": len(TEST), "parity": len(PARITY)})


def load_audio(row):
    audio, _ = librosa.load(ROOT / row["split"] / row["path"], sr=SR)
    return audio[: MAX_SECONDS * SR]


In [ ]:
from transformers import (Seq2SeqTrainer, Seq2SeqTrainingArguments, WhisperForConditionalGeneration,
                          WhisperProcessor)

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="bengali", task="transcribe")


class Clips(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        feats = processor.feature_extractor(load_audio(row), sampling_rate=SR).input_features[0]
        return {"input_features": feats, "labels": processor.tokenizer(row["sentence"]).input_ids}


def collate(items):
    batch = processor.feature_extractor.pad([{"input_features": x["input_features"]} for x in items], return_tensors="pt")
    labels = processor.tokenizer.pad([{"input_ids": x["labels"]} for x in items], return_tensors="pt")
    ids = labels["input_ids"].masked_fill(labels["attention_mask"].ne(1), -100)
    if (ids[:, 0] == model.generation_config.decoder_start_token_id).all():
        ids = ids[:, 1:]                             # the model adds the start token itself
    batch["labels"] = ids
    return batch


def compute_metrics(pred):
    ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
    labels = np.where(pred.label_ids == -100, processor.tokenizer.pad_token_id, pred.label_ids)
    rates = error_rates(processor.batch_decode(labels, skip_special_tokens=True),
                        processor.batch_decode(ids, skip_special_tokens=True))
    return {"wer": rates["wer"], "cer": rates["cer"]}


model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
model.generation_config.language, model.generation_config.task = "bengali", "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.use_cache = False
args = Seq2SeqTrainingArguments(
    output_dir=str(WORK / "checkpoints"), per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    learning_rate=LR, warmup_steps=WARMUP, max_steps=STEPS, gradient_checkpointing=True, fp16=GPU,
    eval_strategy="steps", eval_steps=EVAL_EVERY, save_steps=EVAL_EVERY, save_total_limit=2, logging_steps=25,
    predict_with_generate=True, generation_max_length=128, load_best_model_at_end=True, metric_for_best_model="wer",
    greater_is_better=False, dataloader_num_workers=4 if GPU else 0, remove_unused_columns=False, report_to="none",
    seed=SEED)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=Clips(TRAIN), eval_dataset=Clips(VAL),
                         data_collator=collate, compute_metrics=compute_metrics, processing_class=processor)
started = time.time()
trainer.train()
TRAIN_MINUTES = round((time.time() - started) / 60, 1)
LOG = trainer.state.log_history
VAL_CURVE = [{"step": x["step"], "wer": x["eval_wer"], "cer": x["eval_cer"]} for x in LOG if "eval_wer" in x]
print("train minutes:", TRAIN_MINUTES, "| best checkpoint:", trainer.state.best_model_checkpoint, "| val:", VAL_CURVE)
model = trainer.model
emb = model.model.decoder.embed_tokens.weight
if model.proj_out.weight.data_ptr() != emb.data_ptr():   # Whisper's output layer shares the decoder embedding
    model.proj_out.weight = emb                         # reloading the best checkpoint reported proj_out as missing
print("output layer tied to the decoder embedding:", model.proj_out.weight.data_ptr() == emb.data_ptr())
model.generation_config.language, model.generation_config.task = "bengali", "transcribe"
model.config.use_cache = True
model.to(DTYPE).save_pretrained(str(OUT))           # kept even if a later cell fails
processor.save_pretrained(str(OUT))


In [ ]:
def transcribe(net, rows, batch=32):
    net.eval()
    texts = []
    with ThreadPoolExecutor(4) as pool:
        for i in range(0, len(rows), batch):
            audios = list(pool.map(load_audio, rows[i:i + batch]))
            feats = processor.feature_extractor(audios, sampling_rate=SR, return_tensors="pt").input_features.to(DEVICE, DTYPE)
            with torch.no_grad():
                ids = net.generate(feats, language="bengali", task="transcribe", max_new_tokens=128)
            texts += processor.batch_decode(ids, skip_special_tokens=True)
    return texts


RESULTS = {}
try:   # the test sample is read here, once, after the checkpoint was chosen on dev
    refs = [r["sentence"] for r in TEST]
    tuned = transcribe(model.to(DEVICE, DTYPE), TEST)
    RESULTS["fine_tuned"] = error_rates(refs, tuned)
    base = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL, dtype=DTYPE).to(DEVICE)
    zero_shot = transcribe(base, TEST)
    RESULTS["base_zero_shot"] = error_rates(refs, zero_shot)
    del base
    gc.collect()
    with open(WORK / "predictions_test.jsonl", "w", encoding="utf-8") as f:
        for row, a, b in zip(TEST, tuned, zero_shot):
            f.write(json.dumps({"path": row["path"], "reference": row["sentence"], "fine_tuned": a, "base": b},
                               ensure_ascii=False) + "\n")
except Exception:
    RESULTS["test_error"] = traceback.format_exc()
print(json.dumps(RESULTS, indent=1)[:3000])


In [ ]:
try:   # the app loads a CTranslate2 int8 model with faster-whisper on CPU, so that copy is scored as well
    from huggingface_hub import hf_hub_download
    for name in ("preprocessor_config.json", "tokenizer.json"):   # transformers 5 saves processor_config.json instead;
        if not (OUT / name).exists():                              # training changes neither file, so take the base's
            hf_hub_download(BASE_MODEL, name, local_dir=str(OUT))
    subprocess.run(["ct2-transformers-converter", "--model", str(OUT), "--output_dir", str(CT2), "--quantization", "int8",
                    "--copy_files", "tokenizer.json", "preprocessor_config.json", "--force"], check=True)
    from faster_whisper import WhisperModel
    ct2 = WhisperModel(str(CT2), device="cpu", compute_type="int8", cpu_threads=os.cpu_count() or 2)
    started = time.time()
    # Decoded with librosa like every other clip here: Kaggle's PyAV rejects an argument faster-whisper's decoder passes.
    ct2_texts = ["".join(s.text for s in ct2.transcribe(load_audio(r), language="bn", beam_size=5)[0]).strip()
                 for r in PARITY]
    refs = [r["sentence"] for r in PARITY]
    tuned_same = [json.loads(l)["fine_tuned"] for l in open(WORK / "predictions_test.jsonl", encoding="utf-8")][:len(PARITY)]
    RESULTS["parity"] = {"clips": len(PARITY), "ct2_int8_cpu_beam5": error_rates(refs, ct2_texts),
                         "checkpoint_greedy": error_rates(refs, tuned_same),
                         "ct2_seconds_per_clip": round((time.time() - started) / len(PARITY), 2)}
except Exception:
    RESULTS["parity_error"] = traceback.format_exc()
print(json.dumps(RESULTS.get("parity") or RESULTS.get("parity_error", "")[-1500:], indent=1))


In [ ]:
tuned, base = RESULTS.get("fine_tuned", {}), RESULTS.get("base_zero_shot", {})
par = RESULTS.get("parity", {})
gap = (par["ct2_int8_cpu_beam5"]["wer"] - par["checkpoint_greedy"]["wer"]) if par else None
summary = {
    "wer": {"value": tuned.get("wer"), "base": base.get("wer"), "target": TARGETS["wer"],
            "meets": bool(tuned and base and tuned["wer"] <= TARGETS["wer"] and tuned["wer"] < base["wer"])},
    "cer": {"value": tuned.get("cer"), "base": base.get("cer"), "target": TARGETS["cer"],
            "meets": bool(tuned and base and tuned["cer"] <= TARGETS["cer"] and tuned["cer"] < base["cer"])},
    "ct2_wer_gap": {"value": None if gap is None else round(gap, 4), "target": TARGETS["ct2_wer_gap"],
                    "meets": gap is not None and gap <= TARGETS["ct2_wer_gap"]},
}
report = {"model": "krishi-whisper-bn-v1", "base_model": BASE_MODEL,
          "status": "trained" if GPU else "cpu_smoke_test: a few steps on a few clips; these scores are not results",
          "data": {"dataset": "umongsain/common-voice-13-bengali-normalized", "licence": "CC0 (Mozilla Common Voice)",
                   "train_clips": len(TRAIN), "val_clips": len(VAL), "test_clips": len(TEST), "seed": SEED},
          "training": {"steps": STEPS, "batch": BATCH, "learning_rate": LR, "minutes": TRAIN_MINUTES,
                       "best_checkpoint": trainer.state.best_model_checkpoint, "val_curve": VAL_CURVE},
          "targets": TARGETS, "summary": summary, "results": RESULTS}
manifest = {"python": platform.python_version(), "gpu": torch.cuda.get_device_name(0) if GPU else None,
            "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "librosa": librosa.__version__}}
(WORK / "eval_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
(WORK / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
shutil.rmtree(WORK / "checkpoints", ignore_errors=True)
print(json.dumps(summary, indent=1))
